# Linear-vs-vol backtest grid — summary and answer

The question was: of every way to trade the measured gap between the
FF/ZQ lattice and the SR3 option surface, which strategy family and
config is best — and is anything ALIVE? Everything below is read from the
league and verdicts computed in `linvol_grid_league` /
`linvol_grid_autopsy`; nothing is asserted that a cell above this line
did not produce.

In [1]:
import sys
from pathlib import Path

import pandas as pd

sys.path.append("../../")
sys.path.append(".")
from linvol_grid_common import pick_winner  # noqa: E402

DATA = Path("../data/linvol_grid")
league = pd.read_parquet(DATA / "league.parquet")
verdicts = pd.read_csv(DATA / "verdicts.csv")
real = league[league["family"].isin(["A", "B", "C", "E"])]
live = real[real["n_trades"] > 0]

print("=== verdicts (per-family best, house taxonomy) ===")
print(verdicts.to_string(index=False))

=== verdicts (per-family best, house taxonomy) ===
family                                    best_config  n_trades  gross_bp  net_1x  net_2x  median_cfg_net  dsr               verdict
     A   mode_flank|30-60|thr0.1|momentum|gated=False         5      30.5   -10.9   -52.2           -16.1  0.0 DEAD (too few trades)
     B modal_fly|135-400|thr0.1|long_disp|gated=False        10       7.8    -2.2   -12.2            -3.2  0.0   MARGINAL-maker-only
     C tail_below|60-135|thr0.06|sell_tail|gated=True         5       3.5     1.0    -1.5            -0.6  0.0 DEAD (too few trades)
     E           ics_resid|n/a|thr3.0|fade|gated=True       114      15.9   -98.1  -212.1          -116.9  0.0   MARGINAL-maker-only


In [2]:
w = pick_winner(live)
n_cfg = len(real)
n_live = len(live)
frac_pos_1x = (live["net_1x_bp"] > 0).mean()
frac_pos_2x = (live["net_2x_bp"] > 0).mean()
print(f"grid: {n_cfg} real configs, {n_live} produced trades, "
      f"{frac_pos_1x:.0%} positive at 1x costs, {frac_pos_2x:.0%} at 2x")
print(f"\noverall winner: family {w['family']} | {w['boundary']} | "
      f"dte {w['dte']} | thr {w['thr']} | {w['direction']} | "
      f"gated={w['gated']}")
print(f"  {int(w['n_trades'])} trades, gross {w['total_gross_bp']:+.1f}bp, "
      f"net {w['net_1x_bp']:+.1f} @1x / {w['net_2x_bp']:+.1f} @2x, "
      f"t={w['t_stat']:.2f}")

fam_verdict = verdicts.set_index("family")["verdict"].to_dict()
alive = [f for f, v in fam_verdict.items() if v == "ALIVE"]
print(f"\nfamilies ALIVE: {alive if alive else 'NONE'}")

grid: 272 real configs, 246 produced trades, 11% positive at 1x costs, 5% at 2x

overall winner: family B | modal_fly | dte 135-400 | thr 0.1 | long_disp | gated=False
  10 trades, gross +7.8bp, net -2.2 @1x / -12.2 @2x, t=-0.24

families ALIVE: NONE


In [3]:
print("What the grid establishes, in one place:")
for fam in ("A", "B", "C", "E"):
    g = real[real["family"] == fam]
    gl = g[g["n_trades"] > 0]
    if len(gl):
        print(f"  {fam}: {len(g)} configs, best {gl['net_1x_bp'].max():+.1f}"
              f"bp @1x, median {gl['net_1x_bp'].median():+.1f}bp, "
              f"verdict {fam_verdict.get(fam, 'n/a')}")
    else:
        print(f"  {fam}: {len(g)} configs, no trades")
print("\nPlacebo summary (family A coordinates):")
for tag in ("P1_gauss", "P2_calendar"):
    p = league[league["family"] == tag]
    pl = p[p["n_trades"] > 0]
    print(f"  {tag}: best {pl['net_1x_bp'].max():+.1f}bp, "
          f"median {pl['net_1x_bp'].median():+.1f}bp"
          if len(pl) else f"  {tag}: no trades")

What the grid establishes, in one place:
  A: 192 configs, best +49.5bp @1x, median -20.5bp, verdict DEAD (too few trades)
  B: 48 configs, best +4.5bp @1x, median -3.7bp, verdict MARGINAL-maker-only
  C: 24 configs, best +1.0bp @1x, median -0.6bp, verdict DEAD (too few trades)
  E: 8 configs, best -98.1bp @1x, median -116.9bp, verdict MARGINAL-maker-only

Placebo summary (family A coordinates):
  P1_gauss: best +12.2bp, median -16.1bp
  P2_calendar: best -1.2bp, median -23.4bp
